In [ ]:
# Install kaggle library (runs on Fabric's Spark driver)
import subprocess
subprocess.run(["pip", "install", "kaggle", "--quiet"], check=True)

import os
os.environ["KAGGLE_USERNAME"] = "anusha2005sm"
KAGGLE_KEY = os.environ.get("KAGGLE_KEY")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 11, Finished, Available, Finished, False)

In [10]:
# Cell 1 — only download from Kaggle if files don't exist yet
files = [f.name for f in mssparkutils.fs.ls("Files/")]
if "olist_orders_dataset.csv" not in files:
    import kaggle
    kaggle.api.dataset_download_files(
        "olistbr/brazilian-ecommerce",
        path="/lakehouse/default/Files/",
        unzip=True,
        force=False  # don't overwrite if already exists
    )
    print("Downloaded from Kaggle")
    bronze_path = "Files/"

    for f in mssparkutils.fs.ls(bronze_path):
        if f.name.endswith(".csv"):
            table_name = f.name.replace(".csv", "")
            
            df = spark.read.csv(
                f.path,
                header=True,
                inferSchema=False,   # read everything as string first
                multiLine=True,      # handles newlines inside review text fields
                escape='"'
            )
            
            df.write.format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .saveAsTable(table_name)
            
            print(f"Loaded: {table_name}")
else:
    print("Files already exist — skipping Kaggle download")
    # Always reload Delta tables from latest CSVs in Files/
for f in mssparkutils.fs.ls("Files/"):
    if f.name.endswith(".csv"):
        table_name = f.name.replace(".csv", "")
        
        df = spark.read.csv(
            f.path,
            header=True,
            inferSchema=False,
            multiLine=True,
            escape='"'
        )
        
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)
        
        print(f"Reloaded: {table_name} — {df.count()} rows")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 12, Finished, Available, Finished, False)

Files already exist — skipping Kaggle download
Reloaded: olist_customers_dataset — 99441 rows
Reloaded: olist_geolocation_dataset — 1000163 rows
Reloaded: olist_order_items_dataset — 113500 rows
Reloaded: olist_order_payments_dataset — 104736 rows
Reloaded: olist_order_reviews_dataset — 100074 rows
Reloaded: olist_orders_dataset — 100291 rows
Reloaded: olist_products_dataset — 32951 rows
Reloaded: olist_sellers_dataset — 3095 rows
Reloaded: product_category_name_translation — 71 rows


In [12]:
# Cell 1 — List all tables in your lakehouse
spark.sql("SHOW TABLES").show(truncate=False)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 14, Finished, Available, Finished, False)

+------------------+---------------------------------+-----------+
|namespace         |tableName                        |isTemporary|
+------------------+---------------------------------+-----------+
|Project.Anusha.dbo|olist_customers_dataset          |false      |
|Project.Anusha.dbo|olist_geolocation_dataset        |false      |
|Project.Anusha.dbo|olist_order_items_dataset        |false      |
|Project.Anusha.dbo|olist_order_payments_dataset     |false      |
|Project.Anusha.dbo|olist_order_reviews_dataset      |false      |
|Project.Anusha.dbo|olist_orders_dataset             |false      |
|Project.Anusha.dbo|olist_products_dataset           |false      |
|Project.Anusha.dbo|olist_sellers_dataset            |false      |
|Project.Anusha.dbo|product_category_name_translation|false      |
+------------------+---------------------------------+-----------+



In [13]:
# Store all table names in a list
tables = [row.tableName for row in spark.sql("SHOW TABLES").collect()]
print(tables)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 15, Finished, Available, Finished, False)

['olist_customers_dataset', 'olist_geolocation_dataset', 'olist_order_items_dataset', 'olist_order_payments_dataset', 'olist_order_reviews_dataset', 'olist_orders_dataset', 'olist_products_dataset', 'olist_sellers_dataset', 'product_category_name_translation']


In [14]:
for t in tables:
    # Cell 2 — Schema (column names + data types)
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    print(t)
    print(df.describe().show())


StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 16, Finished, Available, Finished, False)

olist_customers_dataset
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|  count|               99441|               99441|                   99441|              99441|         99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|
| stddev|                NULL|                NULL|       29797.93899620612|               NULL|          NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                   01003|abadia dos dourados|            AC|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|            TO|
+-------+--------------------+--------------------+------------------------+----

In [15]:
from pyspark.sql.functions import col

tables_with_null_values = []

for t in tables:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    print(t)
    print(df.describe().show())
    
    # Check if any column contains null values
    has_nulls = any(df.filter(col(c).isNull()).limit(1).count() > 0 for c in df.columns)
    
    if has_nulls:
        tables_with_null_values.append(t)

print("Tables with null values:", tables_with_null_values)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 17, Finished, Available, Finished, False)

olist_customers_dataset
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|summary|         customer_id|  customer_unique_id|customer_zip_code_prefix|      customer_city|customer_state|
+-------+--------------------+--------------------+------------------------+-------------------+--------------+
|  count|               99441|               99441|                   99441|              99441|         99441|
|   mean|                NULL|                NULL|       35137.47458291851|               NULL|          NULL|
| stddev|                NULL|                NULL|       29797.93899620612|               NULL|          NULL|
|    min|00012a2ce6f8dcda2...|0000366f3b9a7992b...|                   01003|abadia dos dourados|            AC|
|    max|ffffe8b65bbe3087b...|ffffd2657e2aad290...|                   99990|             zortea|            TO|
+-------+--------------------+--------------------+------------------------+----

In [16]:

tables_with_duplicates = []

for t in tables:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
        
       
    # Check for duplicate rows
    if df.count() != df.dropDuplicates().count():
        tables_with_duplicates.append(t)

print("Tables with duplicates:", tables_with_duplicates)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 18, Finished, Available, Finished, False)

Tables with duplicates: ['olist_geolocation_dataset']


In [17]:
for t in tables_with_duplicates:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    
    # Group by all columns and count, keep only groups with more than 1 occurrence
    duplicates = (df.groupBy(df.columns)
                    .count()
                    .filter(col("count") > 1))
    
    print(f"\n--- Duplicates in table: {t} ---")
    duplicates.show(truncate=False)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 19, Finished, Available, Finished, False)


--- Duplicates in table: olist_geolocation_dataset ---
+---------------------------+-------------------+-------------------+----------------+-----------------+-----+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng    |geolocation_city|geolocation_state|count|
+---------------------------+-------------------+-------------------+----------------+-----------------+-----+
|01013                      |-23.547307871775626|-46.634251279358246|sao paulo       |SP               |2    |
|01123                      |-23.527565811818377|-46.638002787119866|sao paulo       |SP               |2    |
|01220                      |-23.542533199001664|-46.64547053996195 |sao paulo       |SP               |2    |
|01254                      |-23.547376320012685|-46.67994319649151 |sao paulo       |SP               |2    |
|01226                      |-23.537195758814683|-46.65149936594724 |sao paulo       |SP               |4    |
|01226                      |-23.538190850683794|-46.651

In [18]:
from functools import reduce

for t in tables_with_null_values:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    
    # Filter rows where at least one column contains a null
    null_rows = df.filter(
        reduce(lambda a, b: a | b, [col(c).isNull() for c in df.columns])
    )
    
    print(f"\n--- Rows with null values in table: {t} ---")
    null_rows.show(truncate=False)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 20, Finished, Available, Finished, False)


--- Rows with null values in table: olist_order_reviews_dataset ---
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|7bc2406110b926393aa56f80a40eba40|73fc7af87114b39712e6da79b0a377eb|4           |NULL                |NULL                                                                                  

In [19]:
from pyspark.sql.functions import count, when, isnan

for t in tables_with_null_values:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    
    total_rows = df.count()
    
    # Build null count for each column
    null_counts = df.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ])
    
    print(f"\n--- Null summary for table: {t} (Total rows: {total_rows}) ---")
    print(f"{'Column':<30} {'Non-Null Count':<20} {'Null Count':<15} {'DType'}")
    print("-" * 80)
    
    null_values = null_counts.collect()[0].asDict()
    
    for c, dtype in df.dtypes:
        null_c = null_values[c]
        non_null = total_rows - null_c
        print(f"{c:<30} {non_null:<20} {null_c:<15} {dtype}")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 21, Finished, Available, Finished, False)


--- Null summary for table: olist_order_reviews_dataset (Total rows: 100074) ---
Column                         Non-Null Count       Null Count      DType
--------------------------------------------------------------------------------
review_id                      100074               0               string
order_id                       100074               0               string
review_score                   100074               0               string
review_comment_title           11568                88506           string
review_comment_message         41827                58247           string
review_creation_date           100074               0               string
review_answer_timestamp        100074               0               string

--- Null summary for table: olist_orders_dataset (Total rows: 100291) ---
Column                         Non-Null Count       Null Count      DType
--------------------------------------------------------------------------------
order_id

In [20]:
df = spark.read.format("delta").load("Tables/dbo/olist_order_reviews_dataset")

# Show the row where review_id is null
print("--- Row with null review_id ---")
df.filter(col("review_id").isNull()).show(truncate=False)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 22, Finished, Available, Finished, False)

--- Row with null review_id ---
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+



In [21]:
from functools import reduce
from pyspark.sql.functions import count, when, col

print("=" * 100)

for t in tables:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    
    total_rows = df.count()
    duplicate_rows = total_rows - df.dropDuplicates().count()
    
    # Header
    print(f"\n TABLE: {t}")
    print(f" Total Rows: {total_rows:,}  |  Duplicate Rows: {duplicate_rows:,}  |  Columns: {len(df.columns)}")
    print("-" * 100)
    print(f"{'Column':<40} {'Non-Null':<15} {'Null Count':<15} {'Null %':<15} {'DType'}")
    print("-" * 100)
    
    # Null counts for all columns in one pass
    null_counts = df.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ]).collect()[0].asDict()
    
    for c, dtype in df.dtypes:
        null_c = null_counts[c]
        non_null = total_rows - null_c
        null_pct = (null_c / total_rows * 100) if total_rows > 0 else 0
        print(f"{c:<40} {non_null:<15,} {null_c:<15,} {null_pct:<14.1f}% {dtype}")
    
    print("=" * 100)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 23, Finished, Available, Finished, False)


 TABLE: olist_customers_dataset
 Total Rows: 99,441  |  Duplicate Rows: 0  |  Columns: 5
----------------------------------------------------------------------------------------------------
Column                                   Non-Null        Null Count      Null %          DType
----------------------------------------------------------------------------------------------------
customer_id                              99,441          0               0.0           % string
customer_unique_id                       99,441          0               0.0           % string
customer_zip_code_prefix                 99,441          0               0.0           % string
customer_city                            99,441          0               0.0           % string
customer_state                           99,441          0               0.0           % string

 TABLE: olist_geolocation_dataset
 Total Rows: 1,000,163  |  Duplicate Rows: 261,831  |  Columns: 5
--------------------------------

In [22]:
#olist_order_reviews_dataset
#Problems : 85 duplicates, 2.1% order id missing - drop it , 1 review id missing - drop, review score msiing - drop, title misiing impute - No title given  message - no message given 
df_reviews = spark.read.format("delta").load("Tables/dbo/olist_order_reviews_dataset")

# 1. Drop duplicates
df_reviews = df_reviews.dropDuplicates()

# 2. Drop rows with null review_id, order_id, review_score
df_reviews = df_reviews.filter(
    col("review_id").isNotNull() & 
    col("order_id").isNotNull() & 
    col("review_score").isNotNull()
)

# 3. Impute nulls in comment fields
df_reviews = df_reviews.fillna({
    "review_comment_title": "No title given",
    "review_comment_message": "No message given"
})

# Verify
total_rows = df_reviews.count()
null_counts = df_reviews.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_reviews.columns
]).collect()[0].asDict()

print(f"Total Rows after cleaning: {total_rows:,}")
print(f"Remaining nulls: { {k: v for k, v in null_counts.items() if v > 0} }")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 24, Finished, Available, Finished, False)

Total Rows after cleaning: 100,074
Remaining nulls: {}


In [23]:
#olist_products_dataset
#where prduct nae is  missing  drop that row, if hiegt, weight wtx=c is misiing impute with a 0
df_products = spark.read.format("delta").load("Tables/dbo/olist_products_dataset")

# 1. Drop rows where product category name is missing
df_products = df_products.filter(col("product_category_name").isNotNull())

# 2. Impute physical dimensions/weight with 0 where missing
df_products = df_products.fillna({
    "product_weight_g": 0,
    "product_length_cm": 0,
    "product_height_cm": 0,
    "product_width_cm": 0
})

# Verify
total_rows = df_products.count()
null_counts = df_products.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_products.columns
]).collect()[0].asDict()

print(f"Total Rows after cleaning: {total_rows:,}")
print(f"Remaining nulls: { {k: v for k, v in null_counts.items() if v > 0} }")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 25, Finished, Available, Finished, False)

Total Rows after cleaning: 32,341
Remaining nulls: {}


In [24]:
from pyspark.sql.functions import expr

df_orders = spark.read.format("delta").load("Tables/dbo/olist_orders_dataset")

# Step 1: Impute from orders table where possible
df_reviews = df_reviews.join(
    df_orders.select("order_id", "order_delivered_customer_date"),
    on="order_id",
    how="left"
).withColumn(
    "review_creation_date",
    when(
        col("review_creation_date").isNull(),
        col("order_delivered_customer_date")
    ).otherwise(col("review_creation_date"))
).drop("order_delivered_customer_date")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 26, Finished, Available, Finished, False)

In [25]:
df_reviews = df_reviews.filter(col("review_answer_timestamp").isNotNull())

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 27, Finished, Available, Finished, False)

In [26]:
cleaned_tables = {
    "olist_order_reviews_dataset": df_reviews,
    "olist_products_dataset": df_products,
    
}

for name, df in cleaned_tables.items():
    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"Tables/dbo/{name}")
    
    print(f"Saved: {name}")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 28, Finished, Available, Finished, False)

Saved: olist_order_reviews_dataset
Saved: olist_products_dataset


In [27]:
from functools import reduce
from pyspark.sql.functions import count, when, col

print("=" * 100)

for t in tables:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    
    total_rows = df.count()
    duplicate_rows = total_rows - df.dropDuplicates().count()
    
    # Header
    print(f"\n TABLE: {t}")
    print(f" Total Rows: {total_rows:,}  |  Duplicate Rows: {duplicate_rows:,}  |  Columns: {len(df.columns)}")
    print("-" * 100)
    print(f"{'Column':<40} {'Non-Null':<15} {'Null Count':<15} {'Null %':<15} {'DType'}")
    print("-" * 100)
    
    # Null counts for all columns in one pass
    null_counts = df.select([
        count(when(col(c).isNull(), c)).alias(c) 
        for c in df.columns
    ]).collect()[0].asDict()
    
    for c, dtype in df.dtypes:
        null_c = null_counts[c]
        non_null = total_rows - null_c
        null_pct = (null_c / total_rows * 100) if total_rows > 0 else 0
        print(f"{c:<40} {non_null:<15,} {null_c:<15,} {null_pct:<14.1f}% {dtype}")
    
    print("=" * 100)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 29, Finished, Available, Finished, False)


 TABLE: olist_customers_dataset
 Total Rows: 99,441  |  Duplicate Rows: 0  |  Columns: 5
----------------------------------------------------------------------------------------------------
Column                                   Non-Null        Null Count      Null %          DType
----------------------------------------------------------------------------------------------------
customer_id                              99,441          0               0.0           % string
customer_unique_id                       99,441          0               0.0           % string
customer_zip_code_prefix                 99,441          0               0.0           % string
customer_city                            99,441          0               0.0           % string
customer_state                           99,441          0               0.0           % string

 TABLE: olist_geolocation_dataset
 Total Rows: 1,000,163  |  Duplicate Rows: 261,831  |  Columns: 5
--------------------------------

In [28]:
for t in tables:
    df = spark.read.format("delta").load(f"Tables/dbo/{t}")
    
    print(f"\n--- {t} ---")
    print(f"Total Rows: {df.count():,} | Total Columns: {len(df.columns)}")
    print(f"{'Column':<40} {'DType'}")
    print("-" * 60)
    
    for c, dtype in df.dtypes:
        print(f"{c:<40} {dtype}")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 30, Finished, Available, Finished, False)


--- olist_customers_dataset ---
Total Rows: 99,441 | Total Columns: 5
Column                                   DType
------------------------------------------------------------
customer_id                              string
customer_unique_id                       string
customer_zip_code_prefix                 string
customer_city                            string
customer_state                           string

--- olist_geolocation_dataset ---
Total Rows: 1,000,163 | Total Columns: 5
Column                                   DType
------------------------------------------------------------
geolocation_zip_code_prefix              string
geolocation_lat                          string
geolocation_lng                          string
geolocation_city                         string
geolocation_state                        string

--- olist_order_items_dataset ---
Total Rows: 113,500 | Total Columns: 7
Column                                   DType
------------------------------------

In [29]:
from pyspark.sql.functions import to_timestamp
df_reviews = spark.read.format("delta").load(f"Tables/dbo/olist_order_reviews_dataset")
df_reviews = spark.sql("""
    SELECT
        review_id,
        order_id,
        CAST(review_score AS INT)                        AS review_score,
        review_comment_title,
        review_comment_message,
        CAST(review_creation_date AS TIMESTAMP)          AS review_creation_date,
        CAST(review_answer_timestamp AS TIMESTAMP)       AS review_answer_timestamp
    FROM olist_order_reviews_dataset
""")

# Verify
df_reviews.printSchema()

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 31, Finished, Available, Finished, False)

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [30]:
from pyspark.sql.functions import to_timestamp, col




# Products — cast dimensions to double for precision
df_products = spark.read.format("delta").load(f"Tables/dbo/olist_products_dataset")
df_products = df_products \
    .withColumn("product_weight_g", col("product_weight_g").cast("double")) \
    .withColumn("product_length_cm", col("product_length_cm").cast("double")) \
    .withColumn("product_height_cm", col("product_height_cm").cast("double")) \
    .withColumn("product_width_cm", col("product_width_cm").cast("double"))

# Verify all schemas
for name, df in [("reviews", df_reviews),  ("products", df_products)]:
    print(f"\n--- {name} ---")
    df.printSchema()

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 32, Finished, Available, Finished, False)


--- reviews ---
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)


--- products ---
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)



In [31]:
%%sql
SELECT *
FROM olist_orders_dataset
WHERE 
    customer_id IS NULL
    OR order_approved_at IS NULL
    OR order_delivered_carrier_date IS NULL
    OR order_delivered_customer_date IS NULL
    OR order_estimated_delivery_date IS NULL
    OR order_id IS NULL
    OR order_purchase_timestamp IS NULL
    OR order_status IS NULL;

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 33, Finished, Available, Finished, False)

<Spark SQL result set with 1000 rows and 8 fields>

In [32]:
%%sql
SELECT DISTINCT(order_status) FROM olist_orders_dataset

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 34, Finished, Available, Finished, False)

<Spark SQL result set with 8 rows and 1 fields>

In [ ]:
import requests

WORKSPACE_ID = ""
DATASET_ID   = ""

token = mssparkutils.credentials.getToken("pbi")

response = requests.post(
    f"https://api.powerbi.com/v1.0/myorg/groups/{WORKSPACE_ID}/datasets/{DATASET_ID}/refreshes",
    headers={"Authorization": f"Bearer {token}"},
    json={"notifyOption": "NoNotification"}
)

print("Refresh triggered" if response.status_code == 202 else f"Failed: {response.status_code} — {response.text}")

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 35, Finished, Available, Finished, False)

Refresh triggered


In [35]:
for f in mssparkutils.fs.ls("Files/"):
    print(f.name, f.size)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 37, Finished, Available, Finished, False)

olist_customers_dataset.csv 9033957
olist_geolocation_dataset.csv 61273883
olist_order_items_dataset.csv 15138581
olist_order_payments_dataset.csv 5784027
olist_order_reviews_dataset.csv 14160298
olist_orders_dataset.csv 17656497
olist_products_dataset.csv 2379446
olist_sellers_dataset.csv 174703
product_category_name_translation.csv 2613
schema.png 209677


In [37]:
df = spark.read.table("olist_orders_dataset")
print(f"Total orders: {df.count()}")

# Show the most recent orders
from pyspark.sql.functions import col
df.orderBy(col("order_purchase_timestamp").desc()).show(5, truncate=False)

StatementMeta(, fc4aa587-7abb-4b24-b937-2241403ace1d, 39, Finished, Available, Finished, False)

Total orders: 100291
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|aafa77c60b3b4aa2926332c6035f7495|0e71d0a457d444ebaeb57aa029757e6b|processing  |2026-05-29 15:14:00     |2026-05-29 16:14:00|2026-05-31 15:14:00         |2026-06-03 15:14:00          |2026-06-05 15:14:00          |
|31d2908b874d47a78f2f0c7c10908df8|1bb67380fcf143c1908a19cda4551b99|shipped     |2026-05-29 15:14:00     |2026-05-29 16: